# Podstawy sieci neuronowych

W tym notebooku:
1. zobaczysz, jak obraz zamienia się w liczby,
2. sprawdzisz, co robią neuron, aktywacja i automatyczne obliczanie gradientu,
3. przygotujesz dane i własny MLP w PyTorchu,
4. wytrenujesz model rozpoznający cyfry **0–9**,
5. przetestujesz **learning rate, SGD / RMSprop / Adam / AdamW, dropout, weight decay, early stopping, inicjalizację i scheduler**.

**Dlaczego takie dane?** Korzystamy z wbudowanego zbioru `sklearn.datasets.load_digits`: 1797 obrazków 8 × 8. Nie potrzeba internetu, pobierania MNIST ani GPU.

**Jak pracować?** Uruchamiaj komórki od góry. Wszystkie pola oznaczone **🔧 ZMIEŃ MNIE** możesz bezpiecznie modyfikować. Niektóre eksperymenty są domyślnie wyłączone, żeby pierwszy przebieg trwał krótko.

> Wymagane pakiety: `torch`, `scikit-learn`, `numpy`, `matplotlib`. W Colab zwykle już są dostępne. Na własnym komputerze można je zainstalować poleceniem `pip install torch scikit-learn numpy matplotlib` (np. w terminalu).

## 1. Obraz to tabela liczb

In [ ]:
import copy
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

# Ustalamy ziarno losowania, aby łatwiej porównywać doświadczenia.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# CPU w zupełności wystarczy dla obrazów 8 × 8.
# Dwa wątki często przyspieszają małe zadania na laptopach.
torch.set_num_threads(2)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Urządzenie:", device)

# Zbiór zawiera odręcznie napisane cyfry i ich poprawne etykiety.
digits = load_digits()
print("Obrazy:", digits.images.shape, "| Etykiety:", digits.target.shape)
print("Zakres pikseli:", digits.images.min(), "–", digits.images.max())

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(11, 5))
for cyfra, ax in enumerate(axes.flat):
    i = np.where(digits.target == cyfra)[0][0]
    ax.imshow(digits.images[i], cmap="magma", vmin=0, vmax=16)
    ax.set_title(f"Etykieta: {cyfra}")
    ax.axis("off")
plt.suptitle("Każdy obrazek ma tylko 8 × 8 pikseli")
plt.tight_layout()
plt.show()

Każdy piksel ma wartość od **0** (ciemno) do **16** (jasno). Sieć MLP nie dostaje obrazka „tak jak my”. Dostaje **64 liczby**. Zobaczmy to na jednym przykładzie.

In [ ]:
nr = 5                        # 🔧 ZMIEŃ MNIE: wybierz inny obraz z zakresu 0–1796
obraz = digits.images[nr]      # Tablica 8 × 8
wektor = obraz.reshape(64)     # 64 liczby w jednym wierszu
print("Kształt obrazu:", obraz.shape)
print("Kształt wektora:", wektor.shape)
print("Pierwszych 16 pikseli:", wektor[:16])
fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.imshow(obraz, cmap="magma", vmin=0, vmax=16)
for i in range(8):
    for j in range(8):
        ax.text(j, i, str(int(obraz[i, j])), ha="center", va="center", fontsize=9,
                color="black" if obraz[i, j] > 11 else "white")
ax.axis("off")
ax.set_title(f"Cyfra {digits.target[nr]} jako 64 liczby")
plt.show()

## 2. Jeden neuron i aktywacja

Neuron oblicza **sumę ważoną** wejść, dodaje przesunięcie (`bias`) i przepuszcza wynik przez funkcję aktywacji:

$$z = w_1x_1 + w_2x_2 + b, \qquad a=\mathrm{ReLU}(z)=\max(0,z).$$

Wagi (`w`) mówią, jak mocno każda cecha wpływa na wynik. **ReLU** wprowadza nieliniowość — dzięki temu sieć z wielu warstw może opisywać zależności bardziej złożone niż jedna prosta.

In [ ]:
# 🔧 ZMIEŃ MNIE: zmień wagę / bias i przewidź wynik przed uruchomieniem.
x = torch.tensor([2.0, 3.0])  # Dwie cechy
w = torch.tensor([0.5, -0.2]) # Waga dla każdej cechy
b = torch.tensor(0.1)         # Przesunięcie

z = (x * w).sum() + b          # Suma: x1*w1 + x2*w2 + b
a = torch.relu(z)             # ReLU: wynik ujemny → 0
print(f"Przed ReLU: {z.item():.3f} | Po ReLU: {a.item():.3f}")

### Zadanie 1
Zmień `b` tak, aby neuron zwrócił **0**, a następnie tak, aby zwrócił **wartość dodatnią**. Co się dzieje, gdy `z` jest ujemne?

## 3. Skąd bierzemy gradient? PyTorch zrobi to za nas

Pochodna określa kierunek i tempo zmiany funkcji. Dla $L(w)=(w-3)^2$ minimum jest przy $w=3$. W PyTorchu `requires_grad=True` oznacza: „śledź operacje, aby później policzyć pochodną”. `backward()` realizuje automatyczne różniczkowanie (autograd).

In [ ]:
waga = torch.tensor(0.0, requires_grad=True)  # Parametr, który chcemy poprawić
strata = (waga - 3) ** 2                     # Funkcja straty L(w)
strata.backward()                            # Liczymy dL/dw
print("L(w) =", strata.item())
print("Gradient dL/dw =", waga.grad.item())

# Jeden krok spadku wzdłuż gradientu: w_new = w_old - learning_rate * gradient.
learning_rate = 0.1                         # 🔧 ZMIEŃ MNIE
with torch.no_grad():                       # Aktualizacja nie buduje nowego grafu gradientów
    waga -= learning_rate * waga.grad
print("Nowa waga:", round(waga.item(), 3))

**Ważne:** gdy w prawdziwej sieci wywołujemy `loss.backward()`, PyTorch propaguje informację o pochodnych od wyjścia do wcześniejszych warstw. To **backpropagation** — nie musimy ręcznie wyprowadzać pochodnej każdej wagi.

## 4. Dane treningowe, walidacyjne i testowe

- **Trening:** na tych przykładach aktualizujemy wagi.
- **Walidacja:** wybieramy ustawienia i moment zatrzymania treningu (bez uczenia na tych danych).
- **Test:** oceniamy wybrany już model, dopiero na końcu.

Podział 70% / 15% / 15%. `stratify=y` dba, żeby w każdym zbiorze było podobnie dużo przykładów każdej cyfry.

In [ ]:
# Piksele zmieniamy z zakresu 0–16 na zakres 0–1.
X = digits.data.astype(np.float32) / 16.0  # (1797, 64)
y = digits.target.astype(np.int64)        # Poprawna cyfra jako liczba 0,...,9

X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, random_state=SEED, stratify=y_tmp)

# TensorDataset paruje każdą tablicę cech z prawidłową odpowiedzią.
train_ds = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
val_ds = TensorDataset(torch.tensor(X_val), torch.tensor(y_val))
test_ds = TensorDataset(torch.tensor(X_test), torch.tensor(y_test))

print("Trening / walidacja / test:", len(train_ds), len(val_ds), len(test_ds))
print("Jeden obraz:", train_ds[0][0].shape, "| Jego etykieta:", train_ds[0][1].item())

## 5. Tworzymy MLP: `nn.Linear`, `nn.ReLU`, `nn.Dropout`

Nasza sieć ma 64 wejścia (piksele), warstwę ukrytą i 10 wyjść (cyfry). Warstwa `nn.Linear(64, 64)` oblicza **$Wx+b$**. `nn.Dropout(p)` losowo wyłącza część aktywacji podczas treningu; podczas testowania nie wyłącza niczego.

Nie dodajemy `softmax` na końcu modelu! Funkcja `nn.CrossEntropyLoss()` oczekuje **surowych wyników (`logits`)** i sama wewnętrznie wykonuje odpowiednią operację numeryczną.

In [ ]:
def zbuduj_mlp(liczba_neuronow=64, dropout=0.10, init="default"):
    # Kolejne operacje wykonywane są po kolei: 64 → neurony ukryte → 10.
    model = nn.Sequential(
        nn.Linear(64, liczba_neuronow),
        nn.ReLU(),
        nn.Dropout(p=dropout),
        nn.Linear(liczba_neuronow, 10),
    )

    # PyTorch domyślnie inicjalizuje wagi. Dodatkowo można wybrać wariant He.
    if init == "he":
        for layer in model:
            if isinstance(layer, nn.Linear):
                nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")
                nn.init.zeros_(layer.bias)
    elif init != "default":
        raise ValueError("init musi mieć wartość 'default' albo 'he'")
    return model

model_probny = zbuduj_mlp(liczba_neuronow=32, dropout=0.15)
print(model_probny)
print("Liczba uczonych parametrów:", sum(p.numel() for p in model_probny.parameters()))

# Sprawdzamy przejście w przód (forward pass) dla 4 obrazków.
batch_x = torch.tensor(X_train[:4])
with torch.no_grad():
    logits = model_probny.eval()(batch_x)
print("Wejście:", batch_x.shape, "→ Wyjście:", logits.shape)
print("Przewidywane cyfry:", logits.argmax(dim=1).tolist())

### Zadanie 2
Zmień liczbę neuronów z 32 na 16, a potem na 128. Jak zmienia się liczba parametrów? Czy zmienia się kształt wyjścia?

## 6. Konfiguracja treningu

Parametry **modelu** (wagi i biasy) są uczone przez algorytm. **Hiperparametry** (np. learning rate, optymalizator, dropout) ustawiamy sami. Poniżej jest jedna komórka do zmieniania parametrów.

In [ ]:
# 🔧 ZMIEŃ MNIE: zmień wartości, a później uruchom komórki z treningiem.
CONFIG = {
    "hidden": 64,              # Liczba neuronów w warstwie ukrytej
    "dropout": 0.10,           # 0.0 = bez dropout, np. 0.30 = mocniejszy dropout
    "optimizer": "AdamW",     # Do wyboru: SGD, RMSprop, Adam, AdamW
    "lr": 0.003,               # Learning rate: np. 0.0001, 0.003, 0.05
    "weight_decay": 0.001,     # Siła ograniczania dużych wag
    "batch_size": 64,          # Rozmiar mini-batcha
    "epochs": 35,              # Maksymalna liczba epok
    "patience": 6,             # Early stopping: ile epok czekamy na poprawę
    "min_delta": 0.0005,       # Minimalna poprawa straty walidacyjnej
    "scheduler_gamma": 0.8,   # Mnożnik LR co 10 epok (1.0 = bez zmiany)
    "init": "default",        # Do wyboru: default, he
}
print(CONFIG)

## 7. Jeden krok treningowy: pięć linijek do zapamiętania

In [ ]:
# Ta komórka tylko DEMONSTRUJE pojedynczy krok; właściwy trening jest poniżej.
demo_model = zbuduj_mlp(liczba_neuronow=16, dropout=0.0).to(device)
demo_opt = torch.optim.SGD(demo_model.parameters(), lr=0.1)
demo_loss = nn.CrossEntropyLoss()
demo_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

x_batch, y_batch = next(iter(demo_loader))
x_batch, y_batch = x_batch.to(device), y_batch.to(device)

demo_model.train()                           # Włącza tryb uczenia (ważne dla dropout)
demo_opt.zero_grad()                        # 1. Zerujemy gradienty z poprzedniego kroku.
predykcje = demo_model(x_batch)            # 2. Forward pass: liczymy wynik sieci.
loss = demo_loss(predykcje, y_batch)        # 3. Porównujemy wynik z poprawną odpowiedzią.
loss.backward()                             # 4. Backprop: liczymy gradient wszystkich wag.
demo_opt.step()                             # 5. Optymalizator aktualizuje wagi.
print("Strata jednego mini-batcha:", round(loss.item(), 4))

## 8. Pełna pętla treningowa: train → validation → early stopping

Ta funkcja obejmuje to, co w praktyce jest potrzebne do trenowania małych modeli:

- mini-batche i losowe mieszanie przykładów,
- obliczanie straty i accuracy,
- wybór optymalizatora,
- opcjonalny **scheduler** (stopniowe zmniejszanie learning rate),
- **early stopping** na walidacji oraz **przywrócenie najlepszych wag**.

**RMSprop, Adam i AdamW** różnią się sposobem wykorzystywania gradientów. AdamW stosuje *decoupled weight decay*, więc jego `weight_decay` nie jest matematycznie identyczny z klasycznym dodaniem regularyzacji L2 do straty.

In [ ]:
def wybierz_optymalizator(model, cfg):
    nazwa = cfg["optimizer"].lower()
    lr, wd = cfg["lr"], cfg["weight_decay"]
    if nazwa == "sgd":
        return torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=wd)
    if nazwa == "rmsprop":
        return torch.optim.RMSprop(model.parameters(), lr=lr, weight_decay=wd)
    if nazwa == "adam":
        return torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    if nazwa == "adamw":
        return torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    raise ValueError("Wybierz SGD, RMSprop, Adam lub AdamW")


def ocen_model(model, loader, criterion):
    model.eval()  # Dropout jest wyłączony; inne warstwy też mogą działać inaczej.
    suma_straty = 0.0
    poprawne = 0
    wszystkie = 0
    with torch.no_grad():  # W walidacji nie liczymy gradientów.
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            suma_straty += loss.item() * len(xb) # Ważymy stratę liczbą przykładów.
            poprawne += (logits.argmax(1) == yb).sum().item()
            wszystkie += len(xb)
    return suma_straty / wszystkie, poprawne / wszystkie


def trenuj(cfg, verbose=True):
    torch.manual_seed(SEED)  # Podobny punkt startowy dla powtarzanych eksperymentów.
    model = zbuduj_mlp(cfg["hidden"], cfg["dropout"], cfg["init"]).to(device)
    criterion = nn.CrossEntropyLoss()  # Przyjmuje logits oraz etykiety 0–9.
    optimizer = wybierz_optymalizator(model, cfg)
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer, step_size=10, gamma=cfg["scheduler_gamma"])
    train_loader = DataLoader(train_ds, batch_size=cfg["batch_size"], shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=256, shuffle=False)
    historia = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "lr": []}
    najlepsza_strata = float("inf")
    najlepsze_wagi = None
    bez_poprawy = 0

    for epoka in range(cfg["epochs"]):
        model.train()  # Włącza dropout tylko na czas treningu.
        suma_straty = 0.0
        poprawne = 0
        wszystkie = 0
        historia["lr"].append(optimizer.param_groups[0]["lr"])

        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()                 # Usuń gradienty z poprzedniego batcha.
            logits = model(xb)                    # Forward pass.
            loss = criterion(logits, yb)           # Cross-entropy: jak bardzo się mylimy?
            loss.backward()                       # Backprop: policz gradienty.
            optimizer.step()                       # Zaktualizuj parametry.

            suma_straty += loss.item() * len(xb)  # Sumowanie dla średniej po PRZYKŁADACH.
            poprawne += (logits.argmax(1) == yb).sum().item()
            wszystkie += len(xb)

        train_loss = suma_straty / wszystkie
        train_acc = poprawne / wszystkie
        val_loss, val_acc = ocen_model(model, val_loader, criterion)
        for key, value in [("train_loss", train_loss), ("train_acc", train_acc),
                           ("val_loss", val_loss), ("val_acc", val_acc)]:
            historia[key].append(value)

        if not np.isfinite(train_loss) or not np.isfinite(val_loss):
            print("Uwaga: strata stała się NaN/Inf. Spróbuj mniejszego learning rate!")
            break

        # Kontrolujemy walidację, a nie trening, żeby nie zapamiętywać danych na pamięć.
        if val_loss < najlepsza_strata - cfg["min_delta"]:
            najlepsza_strata = val_loss
            najlepsze_wagi = copy.deepcopy(model.state_dict())
            bez_poprawy = 0
        else:
            bez_poprawy += 1

        if verbose and (epoka == 0 or (epoka + 1) % 5 == 0):
            print(f"Epoka {epoka+1:02d} | train loss {train_loss:.3f} | "
                  f"val loss {val_loss:.3f} | val accuracy {val_acc:.1%} | "
                  f"lr {historia['lr'][-1]:.5f}")

        if bez_poprawy >= cfg["patience"]:
            if verbose:
                print(f"Early stopping po epoce {epoka+1}: brak poprawy na walidacji.")
            break
        scheduler.step()  # Zmienia LR zgodnie z harmonogramem.

    if najlepsze_wagi is not None:
        model.load_state_dict(najlepsze_wagi)  # Przywracamy model z NAJLEPSZEJ epoki.
    return model, historia

## 9. Trenujemy naszą pierwszą sieć!

In [ ]:
model, historia = trenuj(CONFIG, verbose=True)
print("Ukończono", len(historia["train_loss"]), "epok")

In [ ]:
def rysuj_trening(historia, tytul="Postęp uczenia"):
    epoki = np.arange(1, len(historia["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(epoki, historia["train_loss"], label="trening")
    axes[0].plot(epoki, historia["val_loss"], label="walidacja")
    axes[0].set(xlabel="epoka", ylabel="cross-entropy", title="Strata ↓")
    axes[1].plot(epoki, historia["train_acc"], label="trening")
    axes[1].plot(epoki, historia["val_acc"], label="walidacja")
    axes[1].set(xlabel="epoka", ylabel="accuracy", title="Trafność ↑", ylim=(0, 1.02))
    for ax in axes:
        ax.grid(alpha=0.2)
        ax.legend()
    fig.suptitle(tytul)
    plt.tight_layout()
    plt.show()

rysuj_trening(historia)

**Jak interpretować wykresy?** Jeśli strata treningowa maleje, a walidacyjna zaczyna rosnąć, może występować **overfitting** (przeuczenie). Jeśli obie stoją w miejscu — model może się nie uczyć, np. przez zły learning rate lub błąd w kodzie.

## 10. Ocena na testowych obrazkach — dopiero teraz

In [ ]:
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)
test_loss, test_acc = ocen_model(model, test_loader, nn.CrossEntropyLoss())
print(f"Wynik testowy: accuracy = {test_acc:.1%}, loss = {test_loss:.3f}")

# Przewidujemy etykiety zbioru testowego (bez aktualizowania wag).
model.eval()
with torch.no_grad():
    logits_test = model(torch.tensor(X_test).to(device))
    przewidziane = logits_test.argmax(1).cpu().numpy()

fig, axes = plt.subplots(2, 5, figsize=(11, 5))
for k, ax in enumerate(axes.flat):
    ax.imshow(X_test[k].reshape(8, 8), cmap="magma", vmin=0, vmax=1)
    p = int(przewidziane[k]); t = int(y_test[k])
    ax.set_title(f"Prawda {t} / Model {p}", color="green" if p == t else "crimson")
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Wiersz = prawdziwa cyfra, kolumna = cyfra wskazana przez model.
cm = confusion_matrix(y_test, przewidziane)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm).plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
ax.set_title("Macierz pomyłek — zbiór testowy")
plt.show()

## 11. Laboratorium hiperparametrów

**Każdy wariant uczymy od nowa** i porównujemy na **tej samej walidacji**, bez oglądania testu. Zmieniaj jedną rzecz naraz. Rezultaty zależą od danych i liczby epok; to **ćwiczenie**, nie rzetelny benchmark algorytmów.

Ustaw `WYKONAJ_EKSPERYMENTY = True`, aby uruchomić porównania. Na początku pozostaw `False` dla szybkiego przejścia przez notebook.

In [ ]:
WYKONAJ_EKSPERYMENTY = False   # 🔧 ZMIEŃ MNIE na True

# A. Wpływ learning rate: duże LR nie zawsze oznacza szybszą zbieżność.
if WYKONAJ_EKSPERYMENTY:
    wyniki_lr = {}
    for lr in [0.0001, 0.003, 0.1]:
        cfg = {**CONFIG, "lr": lr, "epochs": 12, "patience": 12}
        _, hist = trenuj(cfg, verbose=False)
        wyniki_lr[lr] = hist
    plt.figure(figsize=(8, 4))
    for lr, hist in wyniki_lr.items():
        plt.plot(hist["val_loss"], label=f"lr = {lr}")
    plt.xlabel("epoka – 1"); plt.ylabel("strata walidacyjna")
    plt.title("Eksperyment A: wpływ learning rate")
    plt.grid(alpha=0.2); plt.legend(); plt.show()

In [ ]:
# B. Optymalizatory — różne mechanizmy aktualizacji wag.
# Dobieramy dla SGD inną wartość LR, żeby porównanie było poglądowe.
if WYKONAJ_EKSPERYMENTY:
    wyniki_opt = {}
    for nazwa, lr in [("SGD", 0.1), ("RMSprop", 0.001),
                      ("Adam", 0.003), ("AdamW", 0.003)]:
        cfg = {**CONFIG, "optimizer": nazwa, "lr": lr,
               "epochs": 14, "patience": 14, "weight_decay": 0.0}
        _, hist = trenuj(cfg, verbose=False)
        wyniki_opt[nazwa] = hist
    plt.figure(figsize=(8, 4))
    for nazwa, hist in wyniki_opt.items():
        plt.plot(hist["val_loss"], label=nazwa)
    plt.title("Eksperyment B: optymalizatory")
    plt.xlabel("epoka – 1"); plt.ylabel("strata walidacyjna")
    plt.grid(alpha=0.2); plt.legend(); plt.show()

In [ ]:
# C. Regularyzacja: dropout i weight decay.
# Warto porównać wartości na TRAIN i VAL; regularyzacja może opóźniać uczenie,
# ale pomagać w radzeniu sobie z nowymi danymi.
if WYKONAJ_EKSPERYMENTY:
    warianty = {
        "bez regularyzacji": (0.0, 0.0),
        "tylko dropout": (0.30, 0.0),
        "tylko weight decay": (0.0, 0.02),
        "oba": (0.30, 0.02),
    }
    plt.figure(figsize=(8, 4))
    for nazwa, (dropout, wd) in warianty.items():
        cfg = {**CONFIG, "dropout": dropout, "weight_decay": wd,
               "epochs": 20, "patience": 20}
        _, hist = trenuj(cfg, verbose=False)
        plt.plot(hist["val_loss"], label=nazwa)
    plt.xlabel("epoka – 1"); plt.ylabel("strata walidacyjna")
    plt.title("Eksperyment C: regularyzacja")
    plt.grid(alpha=0.2); plt.legend(); plt.show()

In [ ]:
# D. Harmonogram LR i inicjalizacja wag.
if WYKONAJ_EKSPERYMENTY:
    cfg_bez = {**CONFIG, "scheduler_gamma": 1.0, "epochs": 25, "patience": 25}
    cfg_z = {**CONFIG, "scheduler_gamma": 0.5, "epochs": 25, "patience": 25}
    _, h_bez = trenuj(cfg_bez, verbose=False)
    _, h_z = trenuj(cfg_z, verbose=False)
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
    axes[0].plot(h_bez["val_loss"], label="bez schedulera")
    axes[0].plot(h_z["val_loss"], label="gamma = 0.5")
    axes[0].set_title("Strata walidacyjna"); axes[0].legend()
    axes[1].plot(h_bez["lr"], label="bez schedulera")
    axes[1].plot(h_z["lr"], label="gamma = 0.5")
    axes[1].set_title("Learning rate"); axes[1].legend()
    plt.tight_layout(); plt.show()

    # 🔧 ZMIEŃ MNIE: spróbuj dodatkowo CONFIG['init'] = 'he'
    # i sprawdź, jak zmieni się pierwszy etap treningu.

## 12. Co zrobić, gdy model się nie uczy?

| Objaw | Możliwa przyczyna | Co sprawdzić? |
|---|---|---|
| `loss` = `NaN` albo gwałtownie rośnie | learning rate za duży | zmniejsz `lr` 10–100 razy |
| `loss` niemal nie spada | LR za mały, problem z danymi lub kodem | porównaj LR; sprawdź `backward()` i `step()` |
| train dobry, validation wyraźnie gorsza | przeuczenie | dropout, weight decay, early stopping, więcej danych |
| wszystko ≈ 10% (losowanie) | brak uczenia / złe etykiety / niewłaściwe wejście | sprawdź etykiety 0–9 i skalę pikseli |
| wyniki bardzo zmienne między próbami | losowa inicjalizacja, mało danych | powtórz eksperyment z kilkoma ziarnami |

**Uwaga:** sam `accuracy` nie mówi wszystkiego — śledź również **stratę** i rozdzielaj dane na trzy zbiory.

## 13. Zadania do samodzielnej pracy

**Poziom ★**
1. Zmień `lr` na `0.0001`, a potem na `0.05`. Ile epok potrzeba do uzyskania np. 90% trafności walidacyjnej? Czy zmiana LR zawsze pomaga?
2. Ustaw `hidden=16`, potem `hidden=128`. Jak zmienia się liczba parametrów, czas i dokładność?
3. Wyłącz `dropout` (`0.0`) oraz `weight_decay` (`0.0`). Porównaj wykresy.

**Poziom ★★**
4. Porównaj `SGD`, `RMSprop`, `Adam`, `AdamW`. Zastanów się, czy uczciwe porównanie powinno używać takiego samego LR dla wszystkich.
5. Zwiększ `epochs` i ustaw `patience=3`. Kiedy algorytm kończy trening? Dlaczego przywracamy najlepsze wagi?
6. Wypróbuj `init='he'` i `scheduler_gamma=0.5`. Czy w tak płytkiej sieci jest duża różnica?

**Poziom ★★★**
7. Dodaj **drugą warstwę ukrytą** (np. `nn.Linear(64,64) → ReLU → nn.Linear(64,32) → ReLU → nn.Linear(32,10)`). Jak musisz zmienić `zbuduj_mlp`?
8. Napisz funkcję wyświetlającą **10 błędnie sklasyfikowanych cyfr** wraz z pewnością softmax. Które cyfry najłatwiej pomylić?
9. Spróbuj sztucznie ograniczyć dane treningowe (np. do 200 przykładów) i sprawdź, czy przeuczenie jest bardziej widoczne.